# 图片提取数据分析

本Notebook用于从图片中提取曲线数据。

## 1. 环境配置与依赖导入

In [ ]:
# 导入标准库
import os
import sys
import datetime
import csv

# 导入数据处理库
import pandas as pd
import numpy as np

# 导入图像处理库
import cv2
import requests
from io import BytesIO

# 导入可视化库
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 导入数据库库
import sqlalchemy
from sqlalchemy import create_engine

# 导入配置
from config import DATABASE_URL, COLOR_TOLERANCE, PROJECT_NAME

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print(f"项目: {PROJECT_NAME}")
print(f"当前时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. 图像处理工具函数定义

In [ ]:
def select_data_region(img):
    """交互式选择数据区域"""
    window_name = '选择数据区域 - 按Enter确认，按c重选'
    cv2.namedWindow(window_name)
    
    while True:
        img_copy = img.copy()
        roi = cv2.selectROI(window_name, img_copy, False, False)
        
        x, y, w, h = roi
        cv2.rectangle(img_copy, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.imshow(window_name, img_copy)
        
        key = cv2.waitKey(0) & 0xFF
        if key == 13:  # Enter键
            cv2.destroyWindow(window_name)
            return roi
        elif key == ord('c'):  # c键重选
            continue
        else:
            cv2.destroyWindow(window_name)
            return roi


def get_axis_range():
    """获取坐标轴范围"""
    print("\n请输入Y轴范围（数值）：")
    y_min = float(input("Y轴最小值: ") or "0")
    y_max = float(input("Y轴最大值: ") or "100")
    
    print("\n请输入X轴范围（日期，格式：YYYY/MM）：")
    x_min = input("X轴起始日期: ") or "2020/01"
    x_max = input("X轴结束日期: ") or "2024/12"
    
    start_date = datetime.datetime.strptime(x_min, '%Y/%m')
    end_date = datetime.datetime.strptime(x_max, '%Y/%m')
    
    return (y_min, y_max), (start_date, end_date)

In [ ]:
def select_sample_point(img):
    """交互式选择曲线的样本点"""
    window_name = '选择曲线样本点 - 左键选择点，ESC清除所有点，Enter确认完成'
    cv2.namedWindow(window_name)
    img_copy = img.copy()
    
    points_data = {'points': []}
    
    def mouse_callback(event, x, y, flags, param):
        nonlocal img_copy
        if event == cv2.EVENT_LBUTTONDOWN:
            param['points'].append((x, y))
            temp_img = img.copy()
            for px, py in param['points']:
                cv2.circle(temp_img, (px, py), 3, (0, 255, 0), -1)
            cv2.imshow(window_name, temp_img)
            img_copy = temp_img
    
    cv2.setMouseCallback(window_name, mouse_callback, points_data)
    cv2.imshow(window_name, img_copy)
    
    while True:
        key = cv2.waitKey(1) & 0xFF
        if key == 13 and points_data['points']:  # Enter键
            cv2.destroyWindow(window_name)
            return points_data['points']
        elif key == 27:  # ESC键清除
            img_copy = img.copy()
            cv2.imshow(window_name, img_copy)
            points_data['points'] = []
    
    return []

In [ ]:
def extract_data_from_graph(image_source, is_url=True, 
                             y_min=0, y_max=100, 
                             start_date=None, end_date=None,
                             color_tolerance=25):
    """
    从图像中提取曲线数据
    
    参数:
        image_source: 图片路径或URL
        is_url: 是否为URL
        y_min, y_max: Y轴数值范围
        start_date, end_date: X轴日期范围
        color_tolerance: 颜色容差
    """
    if start_date is None:
        start_date = datetime.datetime(2020, 1, 1)
    if end_date is None:
        end_date = datetime.datetime(2024, 12, 31)
    
    # 读取图片
    if is_url:
        response = requests.get(image_source)
        img_array = np.asarray(bytearray(response.content), dtype=np.uint8)
    else:
        img_array = np.asarray(bytearray(open(image_source, 'rb').read()), dtype=np.uint8)
    
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    
    print("请框选数据点区域（包含所有数据点的矩形区域）")
    data_roi = select_data_region(img)
    x, y, w, h = data_roi
    
    # 提取数据点区域
    points_region = img[y:y+h, x:x+w]
    
    print("请选择曲线上的多个样本点，按Enter结束选择")
    sample_points = select_sample_point(img)
    
    if not sample_points:
        print("未选择样本点，使用默认颜色检测")
        return [], img
    
    # 获取所有样本点的平均颜色
    sample_colors = [img[point[1], point[0]] for point in sample_points]
    avg_sample_color = np.mean(sample_colors, axis=0)
    print(f"\n样本点平均颜色: {avg_sample_color}")
    
    # 创建颜色掩码
    mask = np.zeros(points_region.shape[:2], dtype=np.uint8)
    
    for i in range(points_region.shape[0]):
        for j in range(points_region.shape[1]):
            pixel_color = points_region[i, j]
            color_diff = np.abs(pixel_color - avg_sample_color)
            if np.all(color_diff <= color_tolerance):
                mask[i, j] = 255
    
    # 形态学操作清理噪点
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=1)
    mask = cv2.erode(mask, kernel, iterations=1)
    
    # 查找轮廓
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # 收集所有点
    points = []
    for contour in contours:
        epsilon = 0.5 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)
        for point in approx:
            px = point[0][0] + x
            py = point[0][1] + y
            points.append((px, py))
    
    # 去重并排序
    points = list(set(points))
    points.sort(key=lambda p: p[0])
    
    # 转换坐标为实际值
    def pixel_to_value(point):
        px, py = point
        
        # Y值映射
        y_value = y_max - (py - y) * (y_max - y_min) / h
        
        # X值（日期）映射
        months_range = (end_date.year - start_date.year) * 12 + end_date.month - start_date.month
        months_from_start = (px - x) * months_range / w
        date = start_date + datetime.timedelta(days=int(months_from_start * 30))
        
        return (date.strftime('%Y/%m'), y_value)
    
    value_points = [pixel_to_value(p) for p in points]
    
    # 可视化结果
    result_img = img.copy()
    cv2.rectangle(result_img, (x, y), (x+w, y+h), (0, 255, 0), 2)
    for px, py in points:
        cv2.circle(result_img, (px, py), 2, (0, 0, 255), -1)
    for px, py in sample_points:
        cv2.circle(result_img, (px, py), 4, (0, 255, 0), -1)
    
    return value_points, result_img

## 3. 简单灰度提取方法

In [ ]:
def extract_gray_curve(image_path, start_date, total_weeks, 
                       y_min, y_max, 
                       min_gray=150, max_gray=200):
    """
    使用灰度值提取曲线数据（简化方法）
    
    参数:
        image_path: 图片路径
        start_date: 起始日期字符串 'YYYY-MM-DD'
        total_weeks: 总周数
        y_min, y_max: Y轴数值范围
        min_gray, max_gray: 灰度值范围
    """
    # 读取图像
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    
    if img is None:
        print(f"无法读取图片: {image_path}")
        return []
    
    height, width = img.shape
    
    # 起始日期
    start = datetime.datetime.strptime(start_date, '%Y-%m-%d')
    
    # 坐标轴的像素范围
    x_axis_start = 0
    x_axis_end = width - 1
    y_axis_start = height - 1
    y_axis_end = 0
    
    # 按周提取数据
    data = []
    for week in range(total_weeks):
        x_coord = int(x_axis_start + week * (x_axis_end - x_axis_start) / total_weeks)
        
        for y_coord in range(height):
            if min_gray <= img[y_coord, x_coord] <= max_gray:
                value = y_min + (y_axis_start - y_coord) * (y_max - y_min) / (y_axis_start - y_axis_end)
                date = (start + datetime.timedelta(weeks=week)).strftime('%Y-%m-%d')
                data.append((date, value))
                break
    
    return data

## 4. 示例：从图片提取数据

In [ ]:
# 示例：使用灰度方法提取数据
# 注意：需要提供实际的图片路径

image_path = 'sample_image.png'  # 替换为实际图片路径

if os.path.exists(image_path):
    data = extract_gray_curve(
        image_path=image_path,
        start_date='2019-01-01',
        total_weeks=300,
        y_min=-1000,
        y_max=2000,
        min_gray=150,
        max_gray=200
    )
    
    print(f"提取到 {len(data)} 个数据点")
    for date, value in data[:10]:
        print(f"Date: {date}, Value: {value:.2f}")
else:
    print(f"图片文件不存在: {image_path}")
    print("请将图片文件放到当前目录并更新image_path变量")
    data = []

## 5. 数据处理与可视化

In [ ]:
# 将提取的数据转换为DataFrame
if data:
    df = pd.DataFrame(data, columns=['dt', 'close'])
    df['dt'] = pd.to_datetime(df['dt'])
    df['区域'] = '默认区域'
    
    # 按日期和区域聚合
    df = df.groupby(['dt', '区域'])['close'].mean().reset_index()
    df.sort_values('dt', inplace=True)
    
    print("数据预览:")
    display(df.head(10))
    
    # 可视化
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(df['dt'], df['close'], marker='o', markersize=2)
    ax.set_xlabel('日期')
    ax.set_ylabel('值')
    ax.set_title('从图片提取的曲线数据')
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    os.makedirs('output', exist_ok=True)
    plt.savefig('output/extracted_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. 数据库操作

In [ ]:
# 数据库连接
def get_db_connection():
    """获取数据库连接"""
    try:
        engine = create_engine(DATABASE_URL, poolclass=sqlalchemy.pool.NullPool)
        conn = engine.connect()
        print("数据库连接成功")
        return engine, conn
    except Exception as e:
        print(f"数据库连接失败: {e}")
        return None, None

engine, conn = get_db_connection()

In [ ]:
# 将数据写入数据库
if data and conn is not None:
    table_name = 'extracted_curve_data'
    try:
        df.to_sql(table_name, con=conn, if_exists='replace', index=False)
        print(f"数据已成功写入表: {table_name}")
        print(f"写入记录数: {len(df)}")
    except Exception as e:
        print(f"写入数据库失败: {e}")

## 7. 结果输出与总结

In [ ]:
# 生成输出报告
print("\n" + "="*60)
print(f"项目: {PROJECT_NAME} - 分析报告")
print("="*60)

if data:
    print(f"\n数据处理完成:")
    print(f"  - 提取数据点数: {len(data)}")
    print(f"  - 日期范围: {data[0][0]} 至 {data[-1][0]}")
    
    # 保存到CSV
    output_file = 'output/extracted_data.csv'
    os.makedirs('output', exist_ok=True)
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Date', 'Value'])
        for date, value in data:
            writer.writerow([date, value])
    print(f"\n数据已保存至: {output_file}")
else:
    print("\n无数据可处理")

print("\n" + "="*60)
print("分析完成！")
print("="*60)

In [ ]:
# 关闭数据库连接
if conn is not None:
    conn.close()
    print("数据库连接已关闭")